# Aula 05 — Laboratório Pandas II — Agrupamentos e Joins

**Disciplina:** Análise e Ciência de Dados
**Curso:** Especialização em Ciência de Dados — IFRN
**Professor:** Plácido Antônio de Souza Neto

---

## Antes de começar

1. Renomeie este arquivo para `aula05_SEUNOME.ipynb`.
2. Mantenha a pasta `dados/` ao lado do notebook, com os arquivos
   `matriculas_ifrn_2026.csv` e `campi_ifrn.xlsx` — **os mesmos da Aula 04**.
3. Responda **na célula indicada**, sem apagar os enunciados.
4. Execute tudo do zero antes de entregar (`Kernel → Restart & Run All`)
   e confira se não há erros.
5. Entregue o `.ipynb` **já executado** (com as saídas visíveis) no Moodle.

> Os dados são **sintéticos**, os mesmos da Aula 04: nomes de campi e cursos
> são reais, mas matrículas, nomes de estudantes e notas foram gerados
> aleatoriamente. Nenhuma conclusão sobre o IFRN pode ser tirada deles.

## Preparação padrão (execute antes de tudo)

```python
import numpy as np
import pandas as pd

df = pd.read_csv("dados/matriculas_ifrn_2026.csv", sep=";", decimal=",",
                 encoding="utf-8", dtype={"matricula": "string"})
df["campus"] = df["campus"].str.strip()
df = df.drop_duplicates().reset_index(drop=True)

campi = pd.read_excel("dados/campi_ifrn.xlsx", sheet_name="campi")
cursos = pd.read_excel("dados/campi_ifrn.xlsx", sheet_name="cursos")
```

## Dicionário de dados

Igual ao da Aula 04 — veja a apostila se precisar relembrar as colunas de
`matriculas_ifrn_2026.csv`. Novidade desta aula:

| Tabela | Colunas |
|---|---|
| `campi` | `campus`, `cidade`, `regiao`, `ano_criacao`, `num_docentes` |
| `cursos` | `curso`, `eixo_tecnologico`, `nivel`, `carga_horaria` |


In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("dados/matriculas_ifrn_2026.csv", sep=";", decimal=",",
                 encoding="utf-8", dtype={"matricula": "string"})
df["campus"] = df["campus"].str.strip()
df = df.drop_duplicates().reset_index(drop=True)

campi = pd.read_excel("dados/campi_ifrn.xlsx", sheet_name="campi")
cursos = pd.read_excel("dados/campi_ifrn.xlsx", sheet_name="cursos")

print(df.shape, campi.shape, cursos.shape)

(480, 12) (12, 5) (10, 4)


## Questão 1 — `groupby` simples (1,0 ponto)

Calcule, a partir de `df` já limpo (ver preparação padrão):

a) a nota média por `turno`;
b) o número de estudantes (`size`) por `modalidade`;
c) explique, em uma frase, a diferença entre usar `.size()` e usar
   `.count()` sobre a coluna `nota_final` dentro de um `groupby`.

In [2]:
media_turno = df.groupby("turno")["nota_final"].mean().round(2)
media_turno.sort_values()

,nota_final
turno,
Noturno,66.76
Matutino,70.90
Vespertino,71.62


In [3]:
estudantes_modalidade = df.groupby("modalidade").size()
estudantes_modalidade.sort_values()

,0
modalidade,
EaD,107
Presencial,373


_Sua resposta escrita:_



Neste passo, ao agrupar com o groupby, será retornado um SeriesGroupBy, então aplicamos uma agregação (média) por turno sobre o objeto e definimos sua precisão como duas casas decimais. Em seguida, ordenamos com o sort sem parâmetros, para obter a lista da menor média para a maior, já que o valor padrão da ordenação é o ascending = True.

Agrupamos os estudantes por modalidade e fazemos a contagem com o size para uma contagem geral de linhas. Ele é mais adequado ao contexto da questão, pois não queremos um detalhamento específico por coluna.

A diferença entre size e count é que o primeiro faz um contagem geral, considerando valores vazios. O segundo faz uma contagem detalhada de valores não nulos por coluna.   

## Questão 2 — `groupby` com múltiplas colunas (1,0 ponto)

a) Agrupe por `campus` e `turno` simultaneamente e calcule a nota média.
b) Que tipo de índice o resultado tem? Use `type(resultado.index)` para
   confirmar.
c) Converta o resultado para uma tabela "larga" (campus nas linhas, turno
   nas colunas) usando `unstack()`.

In [4]:
media_campus_turno = df.groupby(["campus", "turno"])["nota_final"].mean().round(2)
media_campus_turno

campus                   turno     
Apodi                    Matutino      69.89
                         Noturno       71.98
                         Vespertino    73.74
Caicó                    Matutino      67.95
                         Noturno       72.37
                         Vespertino    67.90
Currais Novos            Matutino      75.28
                         Noturno       66.79
                         Vespertino    78.53
João Câmara              Matutino      70.21
                         Noturno       61.79
                         Vespertino    60.97
Macau                    Matutino      70.68
                         Noturno       65.34
                         Vespertino    69.78
Mossoró                  Matutino      74.28
                         Noturno       66.23
                         Vespertino    73.78
Natal-Central            Matutino      68.37
                         Noturno       67.42
                         Vespertino    69.00
Natal-Zona Norte         Matutino      71.52
                         Noturno       67.28
                         Vespertino    76.57
Parnamirim               Matutino      68.02
                         Noturno       64.00
                         Vespertino    74.54
Pau dos Ferros           Matutino      68.35
                         Noturno       65.68
                         Vespertino    76.57
Santa Cruz               Matutino      74.43
                         Noturno       76.49
                         Vespertino    72.91
São Gonçalo do Amarante  Matutino      70.57
                         Noturno       61.93
                         Vespertino    67.31
Name: nota_final, dtype: float64

In [5]:
print(type(media_campus_turno.index))
print(media_campus_turno.index.names)

<class 'pandas.core.indexes.multi.MultiIndex'>
['campus', 'turno']


_Sua resposta escrita:_



Neste passo, primeiro usamos o groupby para criar grupos baseados na combinação das duas categorias. Em seguida, selecionamos nota_final e obtemos a média das notas em cada combinação de campus e turno. Como existem dois níveis de agrupamento, o resultado utiliza um MultiIndex (index composto). Por fim, o unstack reorganiza os níveis dos índices em colunas, deixando campus nas linhas e formando uma tabela no formato expandido.

## Questão 3 — `agg` com named aggregation (1,5 ponto)

Usando `groupby("campus")` e **named aggregation** (a forma
`nome=("coluna", "função")`), produza uma tabela com:

- `n_estudantes` — contagem de matrículas;
- `nota_media` — média de `nota_final`;
- `nota_desvio` — desvio padrão de `nota_final`;
- `taxa_aprovacao` — percentual de `situacao == "Aprovado"` (dica: uma
  função `lambda` que calcula a média de uma comparação booleana, vezes 100).

Ordene o resultado por `taxa_aprovacao`, do maior para o menor.

In [6]:
df.groupby("campus").agg(
    n_estudantes=("matricula", "size"),
    nota_media=("nota_final", "mean"),
    nota_desvio=("nota_final", "std"),
    taxa_aprovacao=("situacao", lambda x: (x == "Aprovado").mean() * 100)
    ).sort_values("taxa_aprovacao", ascending=False).round(
)

,n_estudantes,nota_media,nota_desvio,taxa_aprovacao
campus,,,,
Parnamirim,34,67.0,13.0,71.0
Apodi,34,72.0,14.0,68.0
Natal-Zona Norte,42,70.0,11.0,64.0
Santa Cruz,25,75.0,13.0,64.0
Mossoró,70,72.0,14.0,63.0
Caicó,23,70.0,8.0,61.0
Currais Novos,25,72.0,12.0,60.0
Natal-Central,98,68.0,10.0,56.0
Macau,34,67.0,11.0,56.0


_Sua resposta escrita:_



Nesta etapa, agrupamos pela coluna campus e utilizamos o agg com named aggregation para calcular diferentes estatísticas em uma única tabela. A taxa_aprovacao compara cada valor de situacao com "Aprovado". Como True vale 1 e False vale 0, a média dessa comparação representa a proporção de aprovados, que é multiplicada por 100 para obter o percentual. Por fim, o sort com o ascending como False, que seria o equivalente a um order by DESC no SQL.

## Questão 4 — `pivot_table` (1,5 ponto)

a) Construa uma `pivot_table` com a nota média, `campus` nas linhas e
   `situacao` nas colunas, preenchendo células vazias com `0`
   (`fill_value=0`).
b) Refaça a mesma tabela, agora contando estudantes (em vez de média de
   nota), e adicione uma linha e uma coluna de **total**
   (`margins=True, margins_name="Total"`).
c) Em uma frase: por que faz mais sentido usar contagem, e não média, na
   pivot_table do item (b)?

In [7]:
pivot_df = pd.pivot_table(
    df,
    values="nota_final",
    index="campus",
    columns="situacao",
    aggfunc="mean",
    fill_value=0
)
pivot_df.round(2)

situacao,Aprovado,Reprovado por falta,Reprovado por nota
campus,,,
Apodi,75.29,77.47,51.66
Caicó,72.56,70.58,57.57
Currais Novos,78.03,73.50,54.06
João Câmara,72.23,63.84,53.75
Macau,73.17,63.60,52.37
Mossoró,77.47,65.99,50.64
Natal-Central,71.43,69.19,52.26
Natal-Zona Norte,75.06,69.92,54.74
Parnamirim,73.07,54.75,51.72


In [8]:
df["matricula"].isna().sum()

np.int64(0)

In [9]:
pd.pivot_table(
    df,
    values="matricula",
    index="campus",
    columns="situacao",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

situacao,Aprovado,Em curso,Reprovado por falta,Reprovado por nota,Trancado,Total
campus,,,,,,
Apodi,23,1,3,5,2,34
Caicó,14,1,4,3,1,23
Currais Novos,15,0,4,5,1,25
João Câmara,13,0,7,11,1,32
Macau,19,1,7,6,1,34
Mossoró,44,3,8,10,5,70
Natal-Central,55,3,19,13,8,98
Natal-Zona Norte,27,1,4,8,2,42
Parnamirim,24,0,2,8,0,34


_Sua resposta escrita:_



Neste passo, utilizamos a pivot table para resumir e agregar o dado numérico (média das notas finais) a partir da combinação dos variáveis categóricas campus e situação, preenchendo os valores ausentes com 0 e utilizando a precisão de 2 casas decimais para a apresentação do dado.

Em seguida, utilizamos o mesmo princípio de agregação para contar agrupando a contagem segundo os valores distintos na coluna situação. Adicionamos ainda um total para soma de todos os valores distintos de situação presentes no campus.

A contagem faz mais sentido porque o objetivo é saber quantos estudantes existem nas combinações de campus e situação. e não calcular a média de matrículas.

## Questão 5 — `merge` muitos-para-um (1,5 ponto)

a) Combine `df` com `campi` pela coluna `campus`, com `how="left"` e
   `validate="m:1"`. Chame o resultado de `df_campi`.
b) Combine `df_campi` com `cursos` pela coluna `curso`, do mesmo jeito.
   Chame o resultado de `df_completo`.
c) Confirme que nenhuma linha ficou com `NaN` nas colunas vindas das
   tabelas `campi` ou `cursos` (dica: `.isna().sum()` nas colunas novas).
d) Por que `validate="m:1"` é uma boa prática aqui? O que ela teria
   acusado se `campi` tivesse duas linhas para o mesmo campus?

In [10]:
df_campi = df.merge(campi, on="campus", how="left", validate="m:1")
df_completo = df_campi.merge(cursos, on="curso", how="left", validate="m:1")
print(df_completo[["regiao","eixo_tecnologico"]].isna().sum())

regiao              0
eixo_tecnologico    0
dtype: int64


_Sua resposta escrita:_



Neste passo, foi realizado o relacionamento entre as tabelas de matrículas e campi utilizando o merge, com um left join e a coluna campus como chave de relacionamento, de maneira análoga a uma chave estrangeira (foreign key) no SQL. O left join garante que todas as matrículas presentes no df sejam mantidas, mesmo que não exista uma correspondência na tabela de campi.

O parâmetro "m:1" valida o relacionamento. Nesse caso, o df pode possuir várias ocorrências para um mesmo campus, enquanto a tabela campi deve possuir apenas uma ocorrência para cada campus. Dessa forma, o Pandas verifica se o relacionamento é do tipo Many To One. Caso existam registros duplicados para um mesmo campus na tabela campi, o merge lançará um erro.

Em seguida, foi realizado um segundo relacionamento entre df_campi e a tabela de cursos por meio da coluna curso. Novamente, foi utilizado um left join, garantindo a manutenção de todos os registros de df_campi. O parâmetro "m:1" estabelece que podem existir várias matrículas associadas ao mesmo curso em df_campi, enquanto a tabela cursos deve possuir apenas um registro para cada curso.

Por fim, foi realizada uma verificação de valores ausentes nas colunas regiao e eixo_tecnologico.

## Questão 6 — `how="left"` x `how="inner"` (1,0 ponto)

Crie `cursos_parcial`, removendo de `cursos` as linhas cujo
`eixo_tecnologico` seja `"Informação e Comunicação"`. Em seguida:

a) Faça `df.merge(cursos_parcial, on="curso", how="inner")` e conte as linhas.
b) Faça `df.merge(cursos_parcial, on="curso", how="left")` e conte as linhas.
c) Explique, em uma frase, a diferença entre os dois resultados.

In [11]:
cursos_parcial = cursos[
    cursos["eixo_tecnologico"] != "Informação e Comunicação"
].copy()

In [12]:
df.merge(
    cursos_parcial,
    on="curso",
    how="inner"
).shape

(271, 15)

In [13]:
df.merge(
    cursos_parcial,
    on="curso",
    how="left"
).shape

(480, 15)

_Sua resposta escrita:_



O resultado mostrou uma diferença entre os tipos de join. O inner join resultou em 271 linhas, mantendo apenas os registros que possuem correspondência entre os dois dataframes, considerando a coluna curso utilizada no relacionamento. Já o left join resultou em 480 linhas, pois mantém todos os registros do dataframe da esquerda (df), mesmo quando não existe correspondência do curso em cursos_parcial. Dessa forma, as matrículas são preservadas, e os campos provenientes de cursos_parcial ficam como valores ausentes (NaN) quando não há correspondência.

## Questão 7 — `apply` com `lambda` (1,0 ponto)

a) Usando `df_completo["situacao"].apply(...)` com uma função `lambda`,
   crie a coluna `aprovado` (booleana: `True` se `situacao == "Aprovado"`).
b) Reescreva o mesmo cálculo **sem** `apply` (de forma vetorizada, em uma
   linha). Confirme que os dois resultados são idênticos.
c) Em uma frase: por que a versão vetorizada é preferível na maioria dos
   casos?

In [14]:
df_completo["aprovado"] = df_completo["situacao"].apply(
    lambda x: x == "Aprovado"
)

In [15]:
df_completo["aprovado_vetorizado"] = (
    df_completo["situacao"] == "Aprovado"
)

In [16]:
print(
    df_completo["aprovado"].equals(
        df_completo["aprovado_vetorizado"]
    )
)

True


In [17]:
import timeit

tempo_apply = timeit.timeit(
    lambda: df_completo["situacao"].apply(lambda x: x == "Aprovado"),
    number=100
)

tempo_vetorizado = timeit.timeit(
    lambda: df_completo["situacao"] == "Aprovado",
    number=100
)

print(f"Apply + lambda: {tempo_apply:.6f} segundos")
print(f"Vetorizado:      {tempo_vetorizado:.6f} segundos")


Apply + lambda: 0.203917 segundos
Vetorizado:      0.087463 segundos


_Sua resposta escrita:_



Neste passo, utilizando lambda, o método apply percorre os valores da coluna selecionada e aplica uma função para verificar a condição. Essa abordagem gera uma sobrecarga decorrente das chamadas repetidas de uma função Python para cada elemento da coluna.

Na comparação com a forma vetorizada, observa-se uma diferença de desempenho. Nsendo aproximadamente 2 vezes mais rápida. Isso ocorre porque a operação vetorizada utiliza operações internas otimizadas, evitando a chamada individual da função Python para cada elemento.

Em dataframes menores, essa diferença pode não ser significativa. Entretanto, conforme o volume de dados aumenta, a diferença de desempenho tende a se tornar mais relevante, tornando a abordagem vetorizada preferível.

## Questão 8 — `apply(axis=1)` (1,0 ponto)

Escreva uma função `classificar_carga(linha)` que recebe uma linha de
`df_completo` e devolve:

- `"Curso longo"` se `carga_horaria >= 1800`;
- `"Curso médio"` se `1200 <= carga_horaria < 1800`;
- `"Curso curto"` caso contrário.

Aplique com `df_completo.apply(classificar_carga, axis=1)`, salve em uma
coluna `porte_curso` e mostre `value_counts()`.

In [18]:
def classificar_carga(linha):
    carga = linha["carga_horaria"]
    if carga >= 1800:
        return "Curso longo"
    if carga >= 1200:
        return "Curso médio"
    return "Curso curto"

df_completo["porte_curso"] = df_completo.apply(classificar_carga, axis=1)
print(df_completo["porte_curso"].value_counts())

porte_curso
Curso médio    380
Curso longo     58
Curso curto     42
Name: count, dtype: int64


_Sua resposta escrita:_



Nesta etapa, implementamos o método apply, para executar a função classificar_carga sobre o dado do dataframe, passando a linha inteira como argumento (axis=1), gerando o resultado segundo as condições estabelecidas.

## Questão 9 — Method chaining (1,5 ponto)

Em **uma única cadeia de métodos** (um só `resultado = (...)`, sem
variáveis intermediárias), a partir de `df_completo`:

1. descarte estudantes com `situacao == "Trancado"` (`query`);
2. crie a coluna booleana `aprovado` (`assign`);
3. agrupe por `regiao` e `eixo_tecnologico`;
4. calcule `n` (contagem) e `taxa_aprovacao` (média de `aprovado`) com
   named aggregation;
5. mantenha só grupos com `n >= 10` (`query`);
6. ordene por `taxa_aprovacao` decrescente.

Mostre as 5 primeiras linhas do resultado.

In [19]:
(
    df_completo
    .query('situacao != "Trancado"')
    .assign(aprovado=lambda df: df["situacao"] == "Aprovado")
    .groupby(["regiao", "eixo_tecnologico"])
    .agg(
        n=("aprovado", "size"),
        taxa_aprovacao=("aprovado", "mean")
    )
    .round(2)
    .query("n >= 10")
    .sort_values("taxa_aprovacao", ascending=False)
    .head(5)
)


n  taxa_aprovacao
regiao        eixo_tecnologico                       
Metropolitana Produção Industrial  12            0.92
Oeste         Recursos Naturais    14            0.86
              Gestão e Negócios    19            0.84
Metropolitana Recursos Naturais    21            0.71
Oeste         Produção Industrial  16            0.69

_Sua resposta escrita:_



Nesta etapa, utilizamos o método query com uma expressão de texto para filtrar os registros cuja situação seja diferente de "Trancado". Em seguida, criamos uma nova coluna chamada aprovado, que recebe valores booleanos por meio de uma função lambda, indicando se o estudante foi aprovado.

Depois, agrupamos os dados por regiao e eixo_tecnologico e utilizamos agregações para calcular a quantidade de registros e a taxa de aprovação, obtida pela média da coluna booleana aprovado.

Por fim, arredondamos os resultados, filtramos apenas os grupos que possuem pelo menos 10 registros, ordenamos a taxa de aprovação em ordem decrescente e mostramos as cinco primeiras linhas do resultado.

## Questão 10 — Uma pergunta respondida com dados (1,0 ponto)

> **Estudantes de cursos com carga horária maior têm nota média mais
> alta?**

a) Use `pd.cut()` para criar uma coluna `carga_faixa` a partir de
   `carga_horaria`, com faixas `[0, 1200]`, `(1200, 1800]` e
   `(1800, 3000]`, rotuladas `"Até 1200h"`, `"1201–1800h"` e
   `"Acima de 1800h"`.
b) Agrupe por `carga_faixa` e calcule a nota média (use
   `observed=True` no `groupby` para evitar aviso).
c) Escreva, em uma célula de texto, o que os números mostram — e **uma
   limitação** dessa conclusão (pense em quais outras variáveis diferem
   entre cursos curtos e longos).

In [20]:
df_completo["carga_faixa"] = pd.cut(
    df_completo["carga_horaria"],
    bins=[0, 1200, 1800, 3000],
    labels=["Até 1200h", "1201–1800h", "Acima de 1800h"],
    include_lowest=True
)

In [21]:
media = (
    df_completo
    .groupby("carga_faixa", observed=True)["nota_final"]
    .mean()
    .round(2)
    .reset_index(name="nota_media")
)
media

,carga_faixa,nota_media
0,Até 1200h,71.00
1,1201–1800h,66.86
2,Acima de 1800h,65.15


_Sua resposta escrita:_



Os dados indicam que a média dos estudantes diminiu à medida que a carga horária do curso aumenta. Não dá para afirmar de maneira definitiva que somente a carga horária influencia em uma média maior ou menor. Também podem ser considerados fatores como turno, modalidade do curso e eixo tecnológico para uma análise mais abrangente.

---

## Entrega

- Arquivo: `aula05_SEUNOME.ipynb`, **executado do início ao fim**.
- Prazo e local: tarefa **"Exercício Prático — Aula 05"** no Moodle.
- Critérios: código correto e legível (60%), uso adequado de `groupby`/
  `agg`/`pivot_table`/`merge` (20%), interpretação escrita dos resultados
  (20%).
